# parametrizing attention vectors


# Transformers architecture -- Original

In [6]:
!pip install transformers


In [7]:
from transformers import pipeline, set_seed
generator = pipeline("text-generation", model = 'gpt2')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [8]:
set_seed(23)
generator("I am feeling good", max_length = 20, num_return_sequences = 3)

[{'generated_text': 'I am feeling good about myself, the fact that I am doing well, the fact that I have'},
 {'generated_text': "I am feeling good, I'm feeling good.\n\nI'm feeling good. I'm feeling"},
 {'generated_text': 'I am feeling good. I am getting better and better."\n\nThe report said: "This'}]

In [9]:
# using gpt tokenizer for our own statements

from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
text =" lets test this tokenizer"
encoded_input = tokenizer(text, return_tensors ='pt') #returns pytorch tensors

encoded_input
# note the word lets is token number 8781 in gpt 2


{'input_ids': tensor([[ 8781,  1332,   428, 11241,  7509]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

In [10]:
#lets embede
from transformers import GPT2Model
model = GPT2Model.from_pretrained('gpt2')
output = model(**encoded_input)
output['last_hidden_state'].shape

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

torch.Size([1, 5, 768])

# Bidirectional pre-training with BERT & BART
check readme.md to see  how it woks

 # Fine-tuning a BERT model in PyTorch


In [5]:
import gzip
import shutil
import time
import pandas as pd
import requests
import torch
import torch.nn.functional as F
#import torch.text
import transformers

from transformers import DistilBertTokenizerFast
from transformers import DistilBertForSequenceClassification

torch.backends.cudnn.deterministic = True
RANDOM_SEED = 123
torch.manual_seed(RANDOM_SEED)
Device = torch.device("cuda" if torch.cuda.is_available else 'cpu' )
num_epochs = 3

url = ("https://github.com/rasbt/""machine-learning-book/raw/""main/ch08/movie_data.csv.gz")
filename = url.split("/")[-1]
with open(filename, "wb") as f:
    r = requests.get(url)
    f.write(r.content)
with gzip.open('movie_data.csv.gz', 'rb') as f_in:
    with open('movie_data.csv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)



In [6]:
df = pd.read_csv("movie_data.csv")
df.head()

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0
3,hi for all the people who have seen this wonde...,1
4,"I recently bought the DVD, forgetting just how...",0


In [7]:
train_texts = df.iloc[:35000]['review'].values
train_labels = df.iloc[:35000]['sentiment'].values
valid_texts = df.iloc[35000:40000]['review'].values
valid_labels = df.iloc[35000:40000]['sentiment'].values
test_texts = df.iloc[40000:]['review'].values
test_labels = df.iloc[40000:]['sentiment'].values

# tokenizing the dataset


In [8]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
train_encodings = tokenizer(list(train_texts), padding = True, truncation = True)
valid_encodings = tokenizer(list(valid_texts), padding = True, truncation = True)
test_encodings = tokenizer(list(test_texts), padding = True, truncation = True)

# data preparation

In [9]:
class IMDB(torch.utils.data.Dataset):
  def __init__(self, encodings, lables):
    self.encodings = encodings
    self.labels = lables
  def _get_item_(self, idx):
    item = {key : torch.tensor(values[idx]) for key, values in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx])
    return item
  def __len__(self):
    return len(self.labels)



train_dataset = IMDB(train_encodings, train_labels)
valid_dataset = IMDB(valid_encodings, valid_labels)
test_dataset = IMDB(test_encodings, test_labels)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size = 16, shuffle = True)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size = 16, shuffle = False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size = 16, shuffle = False)



# loading and fine tuning the trained BERT MODEL

In [10]:
# we shall use distilbert  base uncased model similar to bert model but of lighter weight
#model = DistilBertFromSequenceClassification.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')
model.to(Device)
model.train()
optim = torch.optim.Adam(model.parameters(), lr = 5e-5)




config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
class IMDB(torch.utils.data.Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels
  def __getitem__(self, idx):
    item = {key : torch.tensor(values[idx]) for key, values in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx])
    return item
  def __len__(self):
    return len(self.labels)

train_dataset = IMDB(train_encodings, train_labels)
valid_dataset = IMDB(valid_encodings, valid_labels)
test_dataset = IMDB(test_encodings, test_labels)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size = 16, shuffle = True)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size = 16, shuffle = False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size = 16, shuffle = False)

def compute_accuracy(model, data_loader, device):
  with torch.no_grad(): # during evaluation we do not want to update weights
    correct_pred, num_examples = 0, 0
    for batch_idx, batch in enumerate(data_loader):
      input_ids =  batch['input_ids'].to(device) # shoiuld reside on same machine
      attention_mask = batch['attention_mask'].to(device) # shoiuld reside on same machine
      labels = batch['labels'].to(device) # shoiuld reside on same machine
      outputs = model(input_ids, attention_mask = attention_mask) #note we do not pass the logits directly
      logits = outputs['logits']
      predicted_labels = torch.argmax(logits, 1)
      num_examples += labels.size(0)
      correct_pred += (predicted_labels == labels).sum()
    return correct_pred.float() / num_examples * 100

# fine tunings
start_time = time.time()
for epoch in range(num_epochs):
  model.train()
  for batch_idx, batch in enumerate(train_loader):
    input_ids = batch['input_ids'].to(Device)
    attention_mask = batch['attention_mask'].to(Device)
    labels = batch['labels'].to(Device)
    # optimzer.zero_grad()
    # forward pass
    outputs = model(input_ids, attention_mask = attention_mask, labels = labels)
    loss, logits = outputs['loss'], outputs['logits']

    # backward pass
    optim.zero_grad()
    loss.backward()
    optim.step()

    # logging
    if not batch_idx % 250: # print every batch after 250
      print(f'Epoch: {epoch+1:04d}/{num_epochs:04d} | '
            f'Batch {batch_idx:04d}/{len(train_loader):04d} | '
            f'Loss: {loss:.4f}')

  model.eval()  # turn off the drop out and activate batch nomalization
  with torch.set_grad_enabled(False):
    print(f'training accuracy: '
          f'{compute_accuracy(model, train_loader, Device):.2f}%'
          f'\nvalid accuracy: '
          f'{compute_accuracy(model, valid_loader, Device):.2f}%')
    print(f'Time elapsed: {(time.time() - start_time)/60:.2f} min')
    print(f'test accuracy: {compute_accuracy(model, test_loader, Device):.2f}%')

Epoch: 0001/0003 | Batch 0000/2188 | Loss: 0.2447
Epoch: 0001/0003 | Batch 0250/2188 | Loss: 0.0427
Epoch: 0001/0003 | Batch 0500/2188 | Loss: 0.0887
Epoch: 0001/0003 | Batch 0750/2188 | Loss: 0.3475
Epoch: 0001/0003 | Batch 1000/2188 | Loss: 0.1914
Epoch: 0001/0003 | Batch 1250/2188 | Loss: 0.1409
Epoch: 0001/0003 | Batch 1500/2188 | Loss: 0.0939
Epoch: 0001/0003 | Batch 1750/2188 | Loss: 0.2235
Epoch: 0001/0003 | Batch 2000/2188 | Loss: 0.3791
training accuracy: 96.69%
valid accuracy: 92.38%
Time elapsed: 38.72 min
test accuracy: 92.53%
Epoch: 0002/0003 | Batch 0000/2188 | Loss: 0.6297
Epoch: 0002/0003 | Batch 0250/2188 | Loss: 0.0310
Epoch: 0002/0003 | Batch 0500/2188 | Loss: 0.3044
Epoch: 0002/0003 | Batch 0750/2188 | Loss: 0.2380
Epoch: 0002/0003 | Batch 1000/2188 | Loss: 0.0310
Epoch: 0002/0003 | Batch 1250/2188 | Loss: 0.3330
Epoch: 0002/0003 | Batch 1500/2188 | Loss: 0.0903
Epoch: 0002/0003 | Batch 1750/2188 | Loss: 0.3231
Epoch: 0002/0003 | Batch 2000/2188 | Loss: 0.1402
train

# Fine-tuning a transformer more conveniently using the Trainer API